In [3]:
pip install pandas torch

Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install transformers[torch]

Note: you may need to restart the kernel to use updated packages.


In [12]:
import pandas as pd
import torch
data = pd.read_csv('customer_data.csv')

# Define mapping for labels
label_mapping = {'Bronze': 0, 'Silver': 1, 'Gold': 2}  # Assign numbers to each category

# Convert membership_level to numeric labels
data['label'] = data['membership_level'].map(label_mapping)

# Convert labels to PyTorch tensor
labels = torch.tensor(data['label'].tolist())
data['cleaned_text'] = ["Hello, I am a Bronze member!", 
                        "Silver membership offers perks.", 
                        "Gold members get premium benefits.", 
                        "Silver members enjoy discounts.", 
                        "Bronze is the starting tier."]

In [13]:
import re

# Function to clean the text
def clean_text(text):
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'http\S+', '', text)  # Remove URLs
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra whitespaces
    text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
    return text

# Apply cleaning function to your dataset
data['cleaned_text'] = data['text'].apply(clean_text)
print(data['cleaned_text'].head())

0           hello i am a bronze member
1       silver membership offers perks
2    gold members get premium benefits
3       silver members enjoy discounts
4          bronze is the starting tier
Name: cleaned_text, dtype: object


In [14]:
from transformers import BertTokenizer

# Load the BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Tokenize the cleaned text
tokens = tokenizer(
    data['cleaned_text'].tolist(), padding=True, truncation=True, return_tensors='pt', max_length=128
)

print(tokens['input_ids'][:5])  # Check the first 5 tokenized examples

/anaconda/envs/azureml_py38_PT_TF/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


tensor([[  101,  7592,  1045,  2572,  1037,  4421,  2266,   102],
        [  101,  3165,  5779,  4107,  2566,  5705,   102,     0],
        [  101,  2751,  2372,  2131, 12882,  6666,   102,     0],
        [  101,  3165,  2372,  5959, 19575,  2015,   102,     0],
        [  101,  4421,  2003,  1996,  3225,  7563,   102,     0]])


In [15]:
# Check for missing data
print(data.isnull().sum())

# Option 1: Drop rows with missing data
data = data.dropna()

# Option 2: Fill missing values with a placeholder
data['cleaned_text'].fillna('missing', inplace=True)

text                0
membership_level    0
label               0
cleaned_text        0
dtype: int64


/tmp/ipykernel_12190/392484722.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['cleaned_text'].fillna('missing', inplace=True)


In [16]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# Create PyTorch tensors from the tokenized data
input_ids = tokens['input_ids']
attention_masks = tokens['attention_mask']
labels = torch.tensor(data['label'].tolist())

# Create a DataLoader for training
dataset = TensorDataset(input_ids, attention_masks, labels)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

print("DataLoader created successfully!")

DataLoader created successfully!


In [17]:
from sklearn.model_selection import train_test_split

# First, split data into a combined training + validation set and a test set
train_val_inputs, test_inputs, train_val_labels, test_labels = train_test_split(
    input_ids, labels, test_size=0.1, random_state=42
)

# Now, split the combined set into training and validation sets
train_inputs, val_inputs, train_labels, val_labels = train_test_split(
    train_val_inputs, train_val_labels, test_size=0.15, random_state=42
)

# Create DataLoader objects for training, validation, and test sets
train_dataset = TensorDataset(train_inputs, train_labels)
val_dataset = TensorDataset(val_inputs, val_labels)
test_dataset = TensorDataset(test_inputs, test_labels)

train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=16)
test_dataloader = DataLoader(test_dataset, batch_size=16)

print("DataLoader objects for training, validation, and test sets created successfully!")

DataLoader objects for training, validation, and test sets created successfully!
